# Tome C4, Chapter 2: Compression (Python companion)

Exercises first, solutions at the end. We build Huffman codes and beat Shannon–Fano, write an exact arithmetic coder and a universal one that learns the bias of a coin, code the integers à la Elias, implement LZ78 and watch it converge (slowly) to the entropy, and finally try to compress random data.

*Before each solution, predict what the output will look like.*

In [ ]:
import heapq, itertools, math, random, zlib
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
random.seed(2); rng = np.random.default_rng(2)

def H(p):
    return -sum(x * math.log2(x) for x in p if x > 0)

def h(t):
    return H([t, 1 - t])

## Exercises

**P2.1 (★) Huffman.** Implement Huffman's algorithm (returning the codewords). Run it on $0.4,0.2,0.2,0.1,0.1$ (Figure 2.1), and on $0.35,0.17,0.17,0.16,0.15$, where you should also implement the top-down Shannon–Fano split and compare expected lengths. Check the block numbers of Exercise 2.6.

**P2.2 (★★) McMillan.** For random codes (random sets of binary words), test unique decodability with the Sardinas–Patterson algorithm and check that every uniquely decodable one satisfies Kraft's inequality (Theorem 2.1). Find a uniquely decodable code that is not a prefix code.

**P2.3 (★★) Arithmetic coding.** Write an exact arithmetic encoder and decoder with Python fractions, following Theorem 2.5. Reproduce the codeword of $baa$ in Figure 2.2. Encode $1000$ random letters with $p(a)=0.6$, $p(b)=0.3$, $p(c)=0.1$ and compare the length with $\log_2\frac1{p(x^n)}$ and $nH$.

**P2.4 (★★) A coder that learns.** Implement Laplace's rule and check the formula $q(x^n)=\frac{1}{(n+1)\binom nk}$ of Proposition 2.6. For $\theta\in\{0.05,0.2,0.5\}$ and $n=1000$, compare the average code length with $nh(\theta)$ and the bound $nh(\theta)+\log_2(n+1)+2$. Compare with the Krichevsky–Trofimov rule.

**P2.5 (★) Elias codes.** Implement $\gamma$ and $\delta$ (Proposition 2.7), check that they are prefix codes on $1,\dots,10^4$, and plot their lengths against $\log_2 n$.

**P2.6 (★★★) Lempel–Ziv.** Implement LZ78 parsing and encoding, check the examples of Section 2.5 and Exercise 2.8, decode to check correctness, and measure $\ell_{\mathrm{LZ}}/n$ on independent bits with $\theta=0.1$ for $n=10^3,\dots,10^6$. Compare with $h(0.1)$ and with the number of phrases bound of Lemma 2.8.

**P2.7 (★) Random data.** Compress random bytes, English-like text and a compressed file with `zlib`. Check the counting of Proposition 2.11(b): what fraction of the $2^{16}$ words of length $16$ could a compressor shorten by $4$ bits or more?

## Solutions

### P2.1

In [ ]:
def huffman(p):
    heap = [(q, i, {i: ''}) for i, q in enumerate(p)]
    heapq.heapify(heap); cnt = len(p)
    while len(heap) > 1:
        a, _, ca = heapq.heappop(heap); b, _, cb = heapq.heappop(heap)
        merged = {k: '0' + v for k, v in ca.items()}; merged.update({k: '1' + v for k, v in cb.items()})
        heapq.heappush(heap, (a + b, cnt, merged)); cnt += 1
    return [heap[0][2][i] for i in range(len(p))]

def shannon_fano(p):
    idx = sorted(range(len(p)), key=lambda i: -p[i]); code = {i: '' for i in idx}
    def rec(ids):
        if len(ids) <= 1: return
        tot = sum(p[i] for i in ids)
        k = min(range(1, len(ids)), key=lambda k: abs(2 * sum(p[i] for i in ids[:k]) - tot))
        for i in ids[:k]: code[i] += '0'
        for i in ids[k:]: code[i] += '1'
        rec(ids[:k]); rec(ids[k:])
    rec(idx); return [code[i] for i in range(len(p))]

L = lambda p, c: sum(q * len(w) for q, w in zip(p, c))
for p in ([0.4, 0.2, 0.2, 0.1, 0.1], [0.35, 0.17, 0.17, 0.16, 0.15]):
    hc, sc = huffman(p), shannon_fano(p)
    print(p, "\n  Huffman", hc, "L = %.2f" % L(p, hc), "  Shannon-Fano", sc, "L = %.2f" % L(p, sc), "  H = %.3f" % H(p))
for n in (1, 2, 3):
    p = [0.1 ** sum(t) * 0.9 ** (n - sum(t)) for t in itertools.product((0, 1), repeat=n)]
    print("blocks of %d bits: %.3f bits per block, %.4f per bit (h = %.4f)" % (n, L(p, huffman(p)), L(p, huffman(p)) / n, h(0.1)))

### P2.2

In [ ]:
def uniquely_decodable(C):
    C = set(C)
    def dangling(A, B):
        return {b[len(a):] for a in A for b in B if b.startswith(a) and b != a}
    S = dangling(C, C); seen = set()
    while S:
        if '' in S or S & C: return False
        key = frozenset(S)
        if key in seen: return True
        seen.add(key)
        S = dangling(C, S) | dangling(S, C)
    return True

def is_prefix(C):
    return not any(a != b and b.startswith(a) for a in C for b in C)

ud = pref = 0
for _ in range(4000):
    m = random.randint(2, 6)
    C = list({''.join(random.choice('01') for _ in range(random.randint(1, 5))) for _ in range(m)})
    if uniquely_decodable(C):
        ud += 1; pref += is_prefix(C)
        assert sum(2.0 ** -len(w) for w in C) <= 1 + 1e-12
print("uniquely decodable codes found:", ud, " of which prefix codes:", pref, " -- all satisfy Kraft")
print("{0,01,11}: UD =", uniquely_decodable(['0', '01', '11']), " prefix =", is_prefix(['0', '01', '11']))
print("{0,01,10}: UD =", uniquely_decodable(['0', '01', '10']), "  (010 = 0|10 = 01|0)")

### P2.3

In [ ]:
def intervals(p):
    F, c = {}, Fraction(0)
    for s in p: F[s] = c; c += p[s]
    return F

def arith_encode(x, p):
    F = intervals(p); lo, w = Fraction(0), Fraction(1)
    for s in x: lo, w = lo + w * F[s], w * p[s]
    e = max(0, (w.denominator // w.numerator).bit_length() - 1)     # exact ceil(log2(1/w))
    while Fraction(1, 2 ** e) > w: e += 1
    while e > 0 and Fraction(1, 2 ** (e - 1)) <= w: e -= 1
    ell = e + 1
    k = math.ceil(lo * 2 ** ell)
    assert Fraction(k + 1, 2 ** ell) <= lo + w
    return format(k, '0%db' % ell)

def arith_decode(bits, p, n):
    F = intervals(p); t = Fraction(int(bits, 2), 2 ** len(bits)); out = []; lo, w = Fraction(0), Fraction(1)
    for _ in range(n):
        for s in p:
            if lo + w * F[s] <= t < lo + w * (F[s] + p[s]):
                out.append(s); lo, w = lo + w * F[s], w * p[s]; break
    return ''.join(out)

p = {'a': Fraction(6, 10), 'b': Fraction(3, 10), 'c': Fraction(1, 10)}
print("baa ->", arith_encode('baa', p), "  log2 1/p =", round(math.log2(1 / 0.108), 3))
x = ''.join(random.choices('abc', weights=[6, 3, 1], k=1000))
w = arith_encode(x, p)
lp = sum(math.log2(1 / float(p[s])) for s in x)
print("n=1000: code length", len(w), " log2 1/p(x) = %.1f" % lp, " nH = %.1f" % (1000 * H([0.6, 0.3, 0.1])), " decoded ok:", arith_decode(w, p, 1000) == x)

### P2.4

In [ ]:
def seq_code_length(x, rule):
    k, lq = 0, 0.0
    for i, b in enumerate(x):
        q1 = rule(k, i)
        lq += -math.log2(q1 if b else 1 - q1); k += b
    return lq

laplace = lambda k, i: (k + 1) / (i + 2)
kt = lambda k, i: (k + 0.5) / (i + 1)
x = [1, 0, 1, 1, 0, 0, 0, 1, 1, 1]; n, k = len(x), sum(x)
print("Laplace: log2 1/q = %.6f, formula log2((n+1)C(n,k)) = %.6f" % (seq_code_length(x, laplace), math.log2((n + 1) * math.comb(n, k))))
n = 1000
for theta in (0.05, 0.2, 0.5):
    Ls, Kt = [], []
    for _ in range(200):
        x = (rng.random(n) < theta).astype(int)
        Ls.append(seq_code_length(x, laplace) + 2); Kt.append(seq_code_length(x, kt) + 2)
    print("theta=%.2f  n h = %.1f  Laplace %.1f  KT %.1f  bound %.1f" % (theta, n * h(theta), np.mean(Ls), np.mean(Kt), n * h(theta) + math.log2(n + 1) + 2))

The excess over $nh(\theta)$ is close to $\frac12\log_2 n\approx5$ bits for Krichevsky–Trofimov, as Rissanen's theorem predicts, and a little more for Laplace.

### P2.5

In [ ]:
gamma = lambda n: '0' * (n.bit_length() - 1) + format(n, 'b')
delta = lambda n: gamma(n.bit_length()) + format(n, 'b')[1:]

def decode_stream(bits, kind):
    out, i = [], 0
    while i < len(bits):
        z = 0
        while bits[i + z] == '0': z += 1
        v = int(bits[i + z:i + 2 * z + 1], 2); i += 2 * z + 1
        if kind == 'delta':
            v, i = int('1' + bits[i:i + v - 1], 2), i + v - 1
        out.append(v)
    return out

N = list(range(1, 10001)); random.shuffle(N)
for kind, f in (('gamma', gamma), ('delta', delta)):
    assert decode_stream(''.join(f(n) for n in N), kind) == N
    print(kind, "decodes a shuffled stream of 1..10^4 correctly; Kraft sum up to 10^4 = %.5f" % sum(2.0 ** -len(f(n)) for n in range(1, 10001)))
ns = np.arange(1, 5000)
plt.figure(figsize=(5, 3)); plt.plot(ns, [len(gamma(int(n))) for n in ns], label='gamma'); plt.plot(ns, [len(delta(int(n))) for n in ns], label='delta')
plt.plot(ns, np.log2(ns), label='log2 n'); plt.legend(); plt.xscale('log'); plt.show()

### P2.6

In [ ]:
def lz78_parse(x):
    d, out, cur = {'': 0}, [], ''
    for ch in x:
        if cur + ch in d: cur += ch
        else:
            out.append((d[cur], ch)); d[cur + ch] = len(d); cur = ''
    if cur: out.append((d[cur], None))       # repeated last phrase: a pointer only
    return out

def lz78_decode(pairs):
    phrases, out = [''], []
    for i, ch in pairs:
        w = phrases[i] + (ch or ''); out.append(w)
        if ch is not None: phrases.append(w)
    return ''.join(out)

def lz78_length(pairs):
    c = len(pairs)
    return c * (math.ceil(math.log2(c + 1)) + 1)

print(lz78_parse('1011010100010'))
print(lz78_parse('0001101001110000'))
for n in (10 ** 3, 10 ** 4, 10 ** 5, 10 ** 6):
    x = ''.join('1' if b else '0' for b in rng.random(n) < 0.1)
    pr = lz78_parse(x); assert n > 10 ** 5 or lz78_decode(pr) == x
    c = len(pr)
    print("n=%8d  phrases c=%7d  n/log2 n=%8.0f  LZ bits per symbol = %.3f  (h = %.3f)" % (n, c, n / math.log2(n), lz78_length(pr) / n, h(0.1)))

The ratio decreases towards $h(0.1)\approx0.469$, but slowly: the redundancy of LZ78 is of order $\frac{\log\log n}{\log n}$, as the proof of Theorem 2.10 suggests.

### P2.7

In [ ]:
rand = rng.bytes(100000)
text = (' '.join(random.choice(['the', 'of', 'and', 'information', 'channel', 'code', 'entropy', 'bit', 'noise', 'a'])
                 for _ in range(20000))).encode()
for name, data in (("random bytes", rand), ("word salad", text), ("zlib(word salad)", zlib.compress(text, 9))):
    print("%-18s %7d -> %7d bytes" % (name, len(data), len(zlib.compress(data, 9))))
n, k = 16, 4
print("fraction of words of length 16 shortenable by >= 4 bits: at most %.5f (< 2^-3 = 0.125)" % ((2 ** (n - k + 1) - 1) / 2 ** n))

Random bytes grow by a few bytes, structured text shrinks a lot, and compressing the compressed text gains nothing.